# Course 03 lab — Resolve the effective specification

A feature specification is only one input. Resolve organization, platform, domain, project, and feature requirements for Northstar Mutual ticket `AI-1937`. Keep four stages distinct: candidate discovery, applicability resolution, effective-specification resolution, and context packaging. Preserve provenance, surface genuine conflicts, and compose bounded agent context only after the gate is ready.

## Safety boundary

This credential-free lab validates fixture structure and deterministic decision rules. It does **not** authenticate policy publishers or approvers, interpret law, prove catalog completeness, authorize a release, or establish production conformance. Predict each result before executing it.

![Effective specification resolution](assets/effective-specification-resolution.svg)

In [ ]:
import runpy
from dataclasses import replace
from datetime import date
from pathlib import Path

lab = runpy.run_path(Path("lab.py"))
Applicability = lab["Applicability"]
Gate = lab["Gate"]
change, requirements, exceptions = lab["load_scenario"]()
len(requirements), change.id, change.evaluated_on

## 1. Inspect the candidate set

Layer is ownership and scope context—not a universal winner ladder. Inspect decision domain, controlled resource, authority, lifecycle, and control together.

In [ ]:
candidate_view = [
    (item.id, item.layer.name, item.authority.value, item.authority_domain,
     item.resource, item.status.value, item.control)
    for item in requirements
]
candidate_view

## 2. Compare the unsafe concatenation baseline

Retrieval alone cannot explain scope, supersession, authority, conflict, or exceptions. The naive baseline silently presents all 14 statements as equally usable.

In [ ]:
baseline = lab["naive_concatenation"](requirements)
len(baseline), baseline[:3]

## 3. Resolve the reference case

Expect 12 applicable requirements, two non-applicable requirements, no uncertainty, nine effective controls, one valid exception, and a `READY` gate. The counts are diagnostic—not proof that the catalog is complete, authentic, or correct.

In [ ]:
report = lab["resolve_effective_specification"](
    change, requirements, exceptions
)
metrics = lab["resolution_metrics"](report, requirements)
metrics

## 4. Audit applicability evidence

A non-applicable decision needs positive evidence for the mismatch. Unknown input must not be converted into non-applicability.

In [ ]:
applicability_table = [
    (decision.requirement_id, decision.result.value, decision.reason_codes,
     decision.evidence_ids)
    for decision in report.decisions
]
applicability_table

## 5. Failure injection: remove data classification

Predict whether privacy requirements become applicable, non-applicable, or uncertain. A missing fact is not evidence that the condition is false.

In [ ]:
facts_without_classification = tuple(
    fact for fact in change.facts if fact.field != "data_classification"
)
unknown_change = replace(change, facts=facts_without_classification)
unknown_report = lab["resolve_effective_specification"](
    unknown_change, requirements, exceptions
)
unknown_decisions = [
    (item.requirement_id, item.reason_codes)
    for item in unknown_report.decisions
    if item.result is Applicability.UNCERTAIN
]
unknown_report.gate.value, unknown_decisions

## 6. Distinguish false and genuine conflicts

`PRIV-030` and `RET-017` both constrain retention but govern different resources, so both can hold. `PRIV-031` and `RET-017` govern the same final broker record with incompatible values. Remove `PRIV-031` first to prove the false-conflict case; then remove the exception to expose the genuine conflict.

In [ ]:
different_resources = tuple(
    item for item in requirements if item.id != "PRIV-031"
)
compatible = lab["resolve_effective_specification"](
    change, different_resources
)
unresolved = lab["resolve_effective_specification"](change, requirements)
conflict_view = [
    (item.id, item.resource, item.control, item.requirement_ids,
     item.owners, item.action)
    for item in unresolved.conflicts
]
compatible.gate.value, unresolved.gate.value, conflict_view

## 7. Failure injection: expire the scoped exception

Expiry is an enforcement boundary. Structural validation cannot authenticate the training approval locator, but it can refuse a stale record.

In [ ]:
expired_exception = replace(exceptions[0], expires_on=date(2026, 1, 1))
expired_report = lab["resolve_effective_specification"](
    change, requirements, (expired_exception,)
)
expired_report.gate.value, [
    item.reason_codes for item in expired_report.exception_decisions
]

## 8. Authority is not specificity

The feature ticket suggests direct SendGrid delivery. The mandatory platform rule requires the Corporate Messaging Gateway. Inspect the recorded decision rather than assuming platform always wins.

In [ ]:
precedence_view = [
    (item.control, item.selected_requirement_ids,
     item.rejected_requirement_ids, item.reason_code)
    for item in report.precedence
]
precedence_view

## 9. Compose provenance-preserving agent context

Context packaging begins only after a `READY` effective-specification result. It excludes non-applicable and superseded records while retaining governing IDs, source locators, applicability evidence, controlled resources, base obligations, the exception ID, every compensating condition, and optional related-unaffected traceability. An exception changes only its named requirement and explicit scope; every other obligation remains unchanged by default without enumeration. Source locators are structurally complete training data—not authenticated provenance.

In [ ]:
agent_context = lab["compose_agent_context"](report, requirements)
assert "PCI-002" not in agent_context
assert "ARCH-004" not in agent_context
assert "EXC-009" in agent_context
assert "applicability_evidence" in agent_context
assert "related_unaffected_obligation = PRIV-030" in agent_context
print(agent_context)

## 10. Evaluate candidate-discovery recall

Applicability evaluation begins only after candidate discovery. Simulate a selector that misses applicable `AI-012` while retaining both irrelevant records. Precision measures irrelevant inclusion; recall measures dangerous exclusion.

In [ ]:
gold_applicable = {
    item.requirement_id
    for item in report.decisions
    if item.result is Applicability.APPLICABLE
}
selected_candidates = {
    item.id for item in requirements if item.id != "AI-012"
}
selection_metrics = lab["context_selection_metrics"](
    gold_applicable, selected_candidates
)
selection_metrics

## 11. Interpret the evidence honestly

The fixture establishes deterministic behavior for these labelled records: explicit flat-ALL scope, fail-closed uncertainty, lifecycle filtering, domain-bounded authority resolution, controlled-resource conflict detection, scoped exception validation, candidate-selection metrics, and loss-aware context composition. It does not prove legal correctness, complete policy discovery, authentic provenance or approval, implementation conformance, or release readiness.

Continue in the workshop by completing the starter applicability matrix, conflict record, exception review, and effective context before opening the reference solution.

In [ ]:
assert report.gate is Gate.READY
assert compatible.gate is Gate.READY
assert unresolved.gate is Gate.STOP
assert expired_report.gate is Gate.STOP
assert unknown_report.gate is Gate.STOP
assert selection_metrics["recall_percent"] < 100
print("All Course 03 reference and failure-injection checks passed.")